# PlantWise: Beginner Plant-Care Advisor with Strands Decider 2B

**Pitch:** Describe what you see and PlantWise tells you what your plant needs — water, better light, or a closer look.

This notebook demonstrates the **PlantWise** agent built for the Agent Experience Hackathon: Strands Decider.
It shows the reusable decision pattern, runs live conversations through the agent, and presents the
measured benchmark of Strands Decider 2B against an LLM.

**Decision pattern tag:** `#policy`  |  **Decider model:** Strands Decider 2B  |  **LLM:** Amazon Bedrock

---

> Built with the AWS User Group Madurai Strands Decider hackathon kit.

## 1. Setup

We load environment variables from `.env` (which holds the decider URL, API key, and Bedrock model IDs).
**No keys are hardcoded or printed in this notebook** — everything comes from the local `.env` file.

In [ ]:
import json, sys, os
sys.path.insert(0, os.path.abspath('.'))

# Load .env (keys stay in the file, never printed)
from decider_kit.config import load_dotenv, settings
load_dotenv()

s = settings()
print('Decider mode:', s.decider_mode)
print('LLM provider:', s.llm_provider)
print('Bedrock model:', s.bedrock_model)
print('Decider URL :', s.decider_url)
print('API key set :', 'yes' if s.decider_api_key else 'no (will fail)')

## 2. The Decision Pattern: Plant-Care Policy Classification

Before the LLM writes a reply, **Strands Decider 2B** reads the user's plant observations and classifies
the situation into one of three care actions. It also checks whether the situation is urgent.

The typed questions and their option descriptions carry most of the signal:

In [ ]:
from agent.decision import QUESTIONS, PATTERN_TAG, CONFIDENCE_THRESHOLD, PlantCarePolicy
from decider_kit import Decider

print('Pattern tag:', PATTERN_TAG)
print('Confidence threshold (below = ask the user):', CONFIDENCE_THRESHOLD)
print()
print(json.dumps(QUESTIONS, indent=2))

### How the state is built

The agent extracts observation fields (soil moisture, leaf condition, sunlight, plant name) from the
user's free-text message and builds a structured state block that the decider reads:

In [ ]:
from agent.agent import _extract_observation_fields

sample = "My pothos has bone-dry soil and the leaves are drooping badly. It's in a low-light corner."
fields = _extract_observation_fields(sample)
state = PlantCarePolicy.build_state(
    plant_name=fields['plant_name'],
    moisture=fields['moisture'],
    leaves=fields['leaves'],
    sunlight=fields['sunlight'],
    notes=sample,
)
print(state)

## 3. Run the Decider Directly

Let's call Strands Decider 2B on this state and see the raw probabilities and verdict:

In [ ]:
d = Decider()
answers = d.ask(state, QUESTIONS)

care = answers['care_action']
urgent = answers['urgent']

print('=== care_action (choice) ===')
print(f'  Verdict   : {care.label}')
print(f'  Confidence: {care.confidence:.1%}')
if hasattr(care, 'probabilities') and care.probabilities:
    for opt, p in sorted(care.probabilities.items(), key=lambda x: -x[1]):
        print(f'  P({opt:10s}) = {p:.1%}')
print()
print('=== urgent (noul / yes-no) ===')
print(f'  P(urgent)  : {urgent.p_yes:.1%}')
print(f'  Verdict    : {"yes" if urgent.yes else "no"}')
print()
print(f'Decider latency: {d.last_latency_ms:.0f} ms')

## 4. Full Agent Conversation

Now let's run the complete PlantWise agent. The flow is:

1. Extract observation fields from the user's message
2. **Strands Decider 2B** classifies the situation (water / sunlight / inspect / clarify)
3. The verdict is injected into the system prompt
4. **Amazon Bedrock** (via Strands Agents) generates the reply and calls tools

Let's try three turns that exercise each branch of the decision:

In [ ]:
from agent.agent import HackathonAgent, NAME

print('Agent:', NAME)
print('=' * 70)

agent = HackathonAgent()

turns = [
    "My pothos has bone-dry soil and the leaves are drooping. It sits in a dark corner.",
    "My monstera has yellow leaves and gets almost no sunlight near the back wall.",
    "My fern's soil is moist but some leaves are brown at the tips and a few are dropping.",
]

for i, msg in enumerate(turns, 1):
    print(f'\n--- Turn {i} ---')
    print(f'User: {msg}')
    result = agent.chat(msg)
    
    dec = result['decisions'][0] if result['decisions'] else {}
    print(f'Decider verdict : {dec.get("verdict", "N/A")}  (confidence {dec.get("confidence", 0):.0%})')
    print(f'Decider time   : {dec.get("decider_ms", 0)} ms')
    print(f'Urgent         : {dec.get("urgent", False)}')
    print(f'Total latency  : {result["latency_ms"]} ms')
    print(f'Reply          : {result["reply"][:200]}...')
    print()

## 5. The Low-Confidence Path (Clarify)

When the decider's confidence is below 0.55, the agent asks a clarifying question instead of guessing.
Let's see what happens with an ambiguous message:

In [ ]:
ambiguous = "I think something might be wrong with one of my plants but I'm not sure what."
result = agent.chat(ambiguous)

dec = result['decisions'][0] if result['decisions'] else {}
print(f'User           : {ambiguous}')
print(f'Decider verdict : {dec.get("verdict", "N/A")}  (confidence {dec.get("confidence", 0):.0%})')
print(f'Reply          : {result["reply"][:300]}')

## 6. Benchmark Results

The benchmark compares Strands Decider 2B against an LLM (qwen3.5-4b) on the same labelled cases,
answering the same typed questions. Numbers below come from a real run
(`submission/bench/results.json`) on the held-out **test split**.

In [ ]:
bench_path = 'submission/bench/results.json'
with open(bench_path) as f:
    bench = json.load(f)

summary = bench['summary']
d_sum = summary['decider']
l_sum = summary.get('llm', {})

print(f'Decider model : {bench["decider_model"]}')
print(f'Decider mode  : {bench["decider_mode"]}')
print(f'LLM baseline  : {bench.get("llm_model", "N/A")}')
print(f'Cases (test)  : {bench["cases"]}')
print(f'Decisions     : {d_sum["decisions"]}')
print()

rows = [
    ['Metric', 'Strands Decider 2B', bench.get('llm_model', 'LLM')],
    ['Accuracy', f'{d_sum["accuracy"]:.1%}', f'{l_sum.get("accuracy", 0):.1%}'],
    ['Latency p50', f'{d_sum["latency_p50_ms"]:.0f} ms', f'{l_sum.get("latency_p50_ms", 0):.0f} ms'],
    ['Latency p95', f'{d_sum["latency_p95_ms"]:.0f} ms', f'{l_sum.get("latency_p95_ms", 0):.0f} ms'],
    ['Cost / 1k decisions', f'${d_sum["usd_per_1k_decisions"]:.4f}', f'${l_sum.get("usd_per_1k_decisions", 0):.4f}'],
    ['Confident (>=0.7)', f'{d_sum["confident_share"]:.0%}', f'{l_sum.get("confident_share", 0):.0%}'],
    ['Accuracy when confident', f'{d_sum["accuracy_when_confident"]:.1%}', f'{l_sum.get("accuracy_when_confident", 0):.1%}'],
]

col_widths = [max(len(str(r[c])) for r in rows) + 2 for c in range(3)]
for r in rows:
    print(''.join(str(c).ljust(w) for c, w in zip(r, col_widths)))

speedup = l_sum.get('latency_p50_ms', 1) / max(d_sum['latency_p50_ms'], 1)
print(f'\nAt the median the decider was {speedup:.1f}x faster than the LLM.')

### Per-case detail

Each row shows what the decider predicted versus the expected answer, with confidence:

In [ ]:
for i, row in enumerate(bench['rows']):
    exp = row['expected']['care_action']
    pred = row['decider']['care_action']['label']
    conf = row['decider']['care_action']['confidence']
    match = 'OK  ' if exp == pred else 'MISS'
    d_ms = row['decider_ms']
    print(f'{match} | case {i+1:2d} | expected={exp:9s} | got={pred:9s} | conf={conf:.2f} | {d_ms:.0f} ms')

## 7. The Reusable Decision Pattern

The pattern lives in `agent/decision.py` and is tagged `#policy`. It is a three-way classification
with an urgency flag and a low-confidence fallback. To reuse it in your own agent:

1. **Copy** `agent/decision.py` into your project.
2. **Replace** the three options in `QUESTIONS["care_action"]` with your own categories, each with
   a one-line description of what that option means.
3. **Adjust** `CONFIDENCE_THRESHOLD` (default 0.55) for your domain — lower if you want the agent
   to commit more often, higher if you want it to ask more questions.
4. **Call** `PlantCarePolicy` (rename it) from your agent before the LLM runs, and inject the
   verdict into the system prompt.

The pattern works for any domain where you need to route a natural-language input to one of a
small number of actions before generating a response — customer support triage, medical symptom
routing, content moderation — anywhere the classification needs to be fast, cheap and honest
about uncertainty.

In [ ]:
# Show the full decision.py source (the reusable pattern)
import inspect
from agent import decision
print(inspect.getsource(decision))

## 8. Architecture

```
User message
    |
    v
+---------------------------+
| Field extraction (regex)   |   soil moisture, leaf condition, sunlight, plant name
+---------------------------+
    |
    v
+---------------------------+
| Strands Decider 2B         |   one forward pass: care_action + urgent
| (hosted API or local)       |   returns probabilities + confidence
+---------------------------+
    |
    v  (verdict injected into system prompt)
+---------------------------+
| Strands Agent (Bedrock)    |   Amazon Nova Pro generates the reply
| + tools: care tip, log,    |   calls tools based on the verdict
|   history lookup           |
+---------------------------+
    |
    v
Reply to user (+ decision log)
```

**Key design choice:** the decision model classifies; the code decides. Thresholds and actions
are ordinary `if` statements in `agent/decision.py`. The decider returns calibrated probabilities;
the wiring turns them into a verdict the agent can act on.

---

## Summary

| | Strands Decider 2B | LLM baseline |
|---|---|---|
| Role | Classify the observation before the LLM runs | Same classification as a baseline |
| Speed (p50) | ~6 seconds | ~20 seconds |
| Accuracy (test) | 83% | 86% |
| Cost / 1k | $0.035 | $0 (hosted free tier) |
| Confidence | Calibrated probabilities per option | Single answer + self-reported confidence |

The decider is faster on every turn, returns calibrated probabilities the code can branch on,
and costs cents per thousand decisions. When it's not sure, the agent asks a clarifying
question instead of guessing — that's the honest path for a plant-care advisor.

---

Built with the AWS User Group Madurai Strands Decider hackathon kit. See [awsugmdu.in](https://www.awsugmdu.in) for the user group's community and events.